# Item-Item Collaborative Filtering

For each item, we build a vector representation. We can then compare item-item similarities. With these similarities we can compute a predicted score for a user-item pair by looking at all the items that a user has rated and taking the average of his ratings weighted by the similarities between the target item and the user's histroy of rated items.

## Example:

| | Naruto | Bleach | One Piece | Death Note |
|---|---|---|---|---|
| Alice | 9 | 8 | – | 6 |
| Bob | 8 | – | 9 | 5 |
| Carol | – | 9 | 8 | – |
| Dave | 7 | 6 | – | 9 |

### Step 1 — Decide who "liked" what

We call a rating of **7 or higher** a "like". Turning the table above into likes (1) and not-likes (0):

| | Naruto | Bleach | One Piece | Death Note |
|---|---|---|---|---|
| Alice | 1 | 1 | 0 | 0 |
| Bob | 1 | 0 | 1 | 0 |
| Carol | 0 | 1 | 1 | 0 |
| Dave | 1 | 0 | 0 | 1 |

### Step 2 — Represent each anime as a vector

Read a column top-to-bottom and you get a vector of "who liked this anime":

```
Naruto     = [1, 1, 0, 1]   (Alice, Bob, Dave liked it)
Bleach     = [1, 0, 1, 0]   (Alice, Carol liked it)
One Piece  = [0, 1, 1, 0]   (Bob, Carol liked it)
Death Note = [0, 0, 0, 1]   (Dave liked it)
```

### Step 3 — Measure similarity between anime vectors
| | Naruto | Bleach | One Piece | Death Note |
|---|---|---|---|---|
| **Naruto** | 1.00 | 0.41 | 0.41 | 0.58 |
| **Bleach** | 0.41 | 1.00 | 0.50 | 0.00 |
| **One Piece** | 0.41 | 0.50 | 1.00 | 0.00 |
| **Death Note** | 0.58 | 0.00 | 0.00 | 1.00 |

### Step 4 — Predict a rating with a weighted average

```
Carol's ratings: Bleach = 9, One Piece = 8

predicted rating for Naruto
  = (similarity(Naruto, Bleach) × 9 + similarity(Naruto, One Piece) × 8)
    ÷ (similarity(Naruto, Bleach) + similarity(Naruto, One Piece))

  = (0.41 × 9 + 0.41 × 8) / (0.41 + 0.41)
  = (3.69 + 3.28) / 0.82
  ≈ 8.5
```

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter


/Volumes/Kioxia SSD/dam/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
anime = kagglehub.load_dataset(
    KaggleDatasetAdapter.HUGGING_FACE,
    "CooperUnion/anime-recommendations-database",
    "anime.csv",
)

rating = kagglehub.load_dataset(
    KaggleDatasetAdapter.HUGGING_FACE,
    "CooperUnion/anime-recommendations-database",
    "rating.csv",
)

/var/folders/f9/1xlw6hms55d1hz0hg4rqc3q40000gn/T/ipykernel_88317/145285688.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  anime = kagglehub.load_dataset(
/var/folders/f9/1xlw6hms55d1hz0hg4rqc3q40000gn/T/ipykernel_88317/145285688.py:7: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  rating = kagglehub.load_dataset(


In [3]:
anime[0]

{'anime_id': 32281,
 'name': 'Kimi no Na wa.',
 'genre': 'Drama, Romance, School, Supernatural',
 'type': 'Movie',
 'episodes': '1',
 'rating': 9.37,
 'members': 200630}

In [4]:
rating[0]

{'user_id': 1, 'anime_id': 20, 'rating': -1}

## Item-item collaborative filtering

Each item is represented by the users who liked it (rating >= 7). We down-sample to the top 300 most-rated anime and top 1000 most-active users to keep things fast, compute item-item cosine similarity, then predict a user's rating for an item as a similarity-weighted average of their other ratings.

In [5]:
import numpy as np
import pandas as pd

anime_df = anime.to_pandas()
rating_df = rating.to_pandas()

In [6]:
# Build a down-sampled user-item rating matrix and a binary "liked" matrix.
# rating == -1 means "watched but not rated", so drop those first.
LIKE_THRESHOLD = 7
TOP_N_ANIME = 300
TOP_M_USERS = 1000

# Remove rows where user did not provide rating
rated = rating_df[rating_df["rating"] != -1]

# Retrieve TOP_N_ANIME animes with the most number of ratings
top_anime_ids = rated["anime_id"].value_counts().head(TOP_N_ANIME).index
# Filter data to only include the TOP_N_ANIME animes
rated = rated[rated["anime_id"].isin(top_anime_ids)]

# Retrieve the TOP_M_USERS most frequent users 
top_user_ids = rated["user_id"].value_counts().head(TOP_M_USERS).index
# Filter data to only the most frequent users
rated = rated[rated["user_id"].isin(top_user_ids)]

# Reshape table to a user-anime matrix
rating_matrix = rated.pivot(index="user_id", columns="anime_id", values="rating")
# Convert rating to binary like-dislike
liked_matrix = (rating_matrix >= LIKE_THRESHOLD).fillna(False).astype(float)

rating_matrix.shape

(1000, 300)

In [7]:
liked_matrix

anime_id,1,5,6,20,24,30,32,33,43,44,...,28497,28701,28907,28999,29803,30276,30503,31043,31240,31964
user_id,,,,,,,,,,,,,,,,,,,,,
226,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
248,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
294,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
392,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
446,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73340,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0
73378,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
73380,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0


Now each anime is represented by the user's that liked it.

## Test

In [8]:
# Item-item cosine similarity: each item's vector is the column of users who liked it.
V = liked_matrix.to_numpy()
# Compute norms for each user vector
norms = np.linalg.norm(V, axis=0)
# Compute Cosine Similarity
norms[norms == 0] = 1e-9
sim = (V.T @ V) / np.outer(norms, norms) # A (dot) B / |A||B|

sim_df = pd.DataFrame(sim, index=liked_matrix.columns, columns=liked_matrix.columns)
sim_df.shape

(300, 300)

In [20]:
def predict_rating(user_id, anime_id):
    """Predict user's rating for anime_id as a similarity-weighted average of their other ratings."""
    if user_id not in rating_matrix.index or anime_id not in sim_df.columns:
        return np.nan

    # Animes that the user has rated
    user_ratings = rating_matrix.loc[user_id].dropna().drop(anime_id, errors="ignore")
    if user_ratings.empty:
        return np.nan
    # Get similarity scores for the animes that the user has rated against current anime
    sims = sim_df.loc[anime_id, user_ratings.index]
    denom = sims.sum()
    if denom == 0:
        return np.nan

    return (sims * user_ratings).sum() / denom # Weighted Average

Sanity Checks to see if our similarities make sense

In [60]:
name_of.head(15)

anime_id
32281                                       Kimi no Na wa.
5114                      Fullmetal Alchemist: Brotherhood
28977                                             Gintama°
9253                                           Steins;Gate
9969                                         Gintama&#039;
32935    Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...
11061                               Hunter x Hunter (2011)
820                                   Ginga Eiyuu Densetsu
15335    Gintama Movie: Kanketsu-hen - Yorozuya yo Eien...
15417                             Gintama&#039;: Enchousen
4181                                  Clannad: After Story
28851                                       Koe no Katachi
918                                                Gintama
2904                    Code Geass: Hangyaku no Lelouch R2
28891                              Haikyuu!! Second Season
Name: name, dtype: str

In [63]:
#Predict a rating for a given user and anime and show the top-5 anime most similar to a sample anime.
sample_user = rating_matrix.index[0]
sample_anime = 4181 # Clannad Sequel
name_of = anime_df.set_index("anime_id")["name"]
print(f"Predicted rating of '{name_of.get(sample_anime, sample_anime)}' for user_id {sample_user}:", predict_rating(sample_user, sample_anime))

# Look at the top 10 most similar animes
top5 = sim_df[sample_anime].drop(sample_anime).sort_values(ascending=False).head(10)
print(f"\nAnime most similar to {name_of.get(sample_anime, sample_anime)}:")
for aid, score in top5.items():
    print(f"  {name_of.get(aid, aid)}: {score:.3f}")

Predicted rating of 'Clannad: After Story' for user_id 226: 7.855893058378239

Anime most similar to Clannad: After Story:
  Clannad: 0.939
  Toradora!: 0.880
  Code Geass: Hangyaku no Lelouch: 0.875
  Steins;Gate: 0.873
  Angel Beats!: 0.873
  Code Geass: Hangyaku no Lelouch R2: 0.866
  Bakemonogatari: 0.855
  Ano Hi Mita Hana no Namae wo Bokutachi wa Mada Shiranai.: 0.855
  Shingeki no Kyojin: 0.850
  No Game No Life: 0.847


The most similar anime is the prequel and the second most similar is also a romance show

In [66]:
def recommend_top_n(user_id, n=10):
    """Recommend the top-n anime for a user by predicting a score for every anime they haven't rated."""
    if user_id not in rating_matrix.index:
        return pd.Series(dtype=float)

    user_ratings = rating_matrix.loc[user_id].dropna()
    candidates = sim_df.columns.difference(user_ratings.index)

    sims = sim_df.loc[candidates, user_ratings.index]
    denom = sims.abs().sum(axis=1)
    scores = pd.Series(sims.values @ user_ratings.values, index=candidates) / denom

    return scores.dropna().sort_values(ascending=False).head(n)

In [67]:
# Demo: top-10 recommendations for a sample user.
top_n = recommend_top_n(sample_user, n=10)
print(f"Top recommendations for user {sample_user}:")
for aid, score in top_n.items():
    print(f"  {name_of.get(aid, aid)}: {score:.2f}")

Top recommendations for user 226:
  Boku no Hero Academia: 7.89
  Gate: Jieitai Kanochi nite, Kaku Tatakaeri: 7.88
  Shokugeki no Souma: 7.88
  Golden Time: 7.88
  Highschool of the Dead: Drifters of the Dead: 7.88
  Re:Zero kara Hajimeru Isekai Seikatsu: 7.88
  Aldnoah.Zero: 7.87
  Btooom!: 7.87
  Boku dake ga Inai Machi: 7.87
  Ao Haru Ride: 7.87


## Recommend for a real MyAnimeList user

Scrape a public MAL profile's scored anime list , then use our similarity matrix to determine the top animes for the profile

In [74]:
import requests
import time

def scrape_mal_ratings(username):
    """Fetch a public MyAnimeList user's scored anime as a Series of {anime_id: score}."""
    entries = []
    offset = 0
    while True:
        resp = requests.get(
            f"https://myanimelist.net/animelist/{username}/load.json",
            params={"offset": offset, "status": 7},  # status=7 -> all list statuses
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=10,
        )
        resp.raise_for_status()
        batch = resp.json()
        entries.extend(batch)
        if len(batch) < 300:
            break
        offset += 300
        time.sleep(0.5)  # be polite between pages

    return pd.Series(
        {e["anime_id"]: e["score"] for e in entries if e["score"] > 0},
        name="rating",
    )

In [75]:
def recommend_for_mal_user(username, n=10):
    """Scrape a MAL user's ratings and recommend anime using the item-item similarity model."""
    user_ratings = scrape_mal_ratings(username)
    user_ratings = user_ratings[user_ratings.index.isin(sim_df.columns)]
    if user_ratings.empty:
        raise ValueError(f"No overlap between {username}'s ratings and the top-{TOP_N_ANIME} anime subset")

    candidates = sim_df.columns.difference(user_ratings.index)
    sims = sim_df.loc[candidates, user_ratings.index]
    denom = sims.abs().sum(axis=1)
    scores = pd.Series(sims.values @ user_ratings.values, index=candidates) / denom

    return scores.dropna().sort_values(ascending=False).head(n)

In [76]:
# Demo: recommend for a real public MAL profile (swap in any public username).
username = "Kyoushiii"
recs = recommend_for_mal_user(username, n=20)
print(f"Recommendations for MAL user '{username}':")
for aid, score in recs.items():
    print(f"  {name_of.get(aid, aid)}: {score:.2f}")

Recommendations for MAL user 'Kyoushiii':
  Serial Experiments Lain: 8.07
  Paprika: 8.07
  Majo no Takkyuubin: 8.06
  Junjou Romantica: 8.06
  Nana: 8.06
  Kaze no Tani no Nausicaä: 8.06
  Gake no Ue no Ponyo: 8.06
  Neon Genesis Evangelion: The End of Evangelion: 8.06
  Tonari no Totoro: 8.05
  Ghost in the Shell: 8.05
  Akira: 8.05
  Ookami Kodomo no Ame to Yuki: 8.05
  Kotonoha no Niwa: 8.05
  Nichijou: 8.05
  Hotarubi no Mori e: 8.05
  Cowboy Bebop: Tengoku no Tobira: 8.05
  Mononoke Hime: 8.05
  Azumanga Daioh: 8.04
  Toki wo Kakeru Shoujo: 8.04
  Shinsekai yori: 8.04
